In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    else:
        df['ts'] = pd.NaT

# Helper to create compact summary for a dataframe
def compact_summary(df, kpi_col=None, sample_col=None, cmdb_limit=50, kpi_limit=20):
    total_rows = len(df)
    cmdb_vals = df['cmdb_id'].dropna().unique().tolist()
    cmdb_trunc = cmdb_vals[:cmdb_limit]
    unique_cmdb_count = len(cmdb_vals)
    earliest = df['ts'].min() if total_rows>0 else pd.NaT
    latest = df['ts'].max() if total_rows>0 else pd.NaT

    kpi_counts = None
    if kpi_col and kpi_col in df.columns:
        kpi_counts = df[kpi_col].fillna("<NA>").value_counts().head(kpi_limit).reset_index()
        kpi_counts.columns = [kpi_col, 'count']

    sample_counts = None
    if sample_col and sample_col in df.columns:
        sample_counts = df[sample_col].fillna("<NA>").value_counts().head(kpi_limit).reset_index()
        sample_counts.columns = [sample_col, 'count']

    return {
        'total_rows': total_rows,
        'unique_cmdb_count': unique_cmdb_count,
        'unique_cmdb_ids_truncated': cmdb_trunc,
        'earliest_ts_utc': earliest,
        'latest_ts_utc': latest,
        'kpi_counts': kpi_counts,
        'sample_counts': sample_counts
    }

# Build summaries
metric_summary = compact_summary(metric_df, kpi_col='kpi_name')
trace_summary = compact_summary(trace_df, kpi_col='trace_name')
log_summary = compact_summary(log_df, kpi_col='log_name')
error_logs_summary = compact_summary(error_logs_df, sample_col='message')

# Create a compact summary table (one row per file) for top-level fields
summary_table = pd.DataFrame([
    {
        'file': 'metric.csv',
        'total_rows': metric_summary['total_rows'],
        'unique_cmdb_count': metric_summary['unique_cmdb_count'],
        'unique_cmdb_ids_truncated': metric_summary['unique_cmdb_ids_truncated'],
        'earliest_ts_utc': metric_summary['earliest_ts_utc'],
        'latest_ts_utc': metric_summary['latest_ts_utc']
    },
    {
        'file': 'trace.csv',
        'total_rows': trace_summary['total_rows'],
        'unique_cmdb_count': trace_summary['unique_cmdb_count'],
        'unique_cmdb_ids_truncated': trace_summary['unique_cmdb_ids_truncated'],
        'earliest_ts_utc': trace_summary['earliest_ts_utc'],
        'latest_ts_utc': trace_summary['latest_ts_utc']
    },
    {
        'file': 'log.csv',
        'total_rows': log_summary['total_rows'],
        'unique_cmdb_count': log_summary['unique_cmdb_count'],
        'unique_cmdb_ids_truncated': log_summary['unique_cmdb_ids_truncated'],
        'earliest_ts_utc': log_summary['earliest_ts_utc'],
        'latest_ts_utc': log_summary['latest_ts_utc']
    },
    {
        'file': 'error_logs.csv',
        'total_rows': error_logs_summary['total_rows'],
        'unique_cmdb_count': error_logs_summary['unique_cmdb_count'],
        'unique_cmdb_ids_truncated': error_logs_summary['unique_cmdb_ids_truncated'],
        'earliest_ts_utc': error_logs_summary['earliest_ts_utc'],
        'latest_ts_utc': error_logs_summary['latest_ts_utc']
    }
])

# Prepare KPI/sample top lists (top up to 20)
metric_kpis = metric_summary['kpi_counts'] if metric_summary['kpi_counts'] is not None else pd.DataFrame(columns=['kpi_name','count'])
trace_kpis = trace_summary['kpi_counts'] if trace_summary['kpi_counts'] is not None else pd.DataFrame(columns=['trace_name','count'])
log_kpis = log_summary['kpi_counts'] if log_summary['kpi_counts'] is not None else pd.DataFrame(columns=['log_name','count'])
error_messages = error_logs_summary['sample_counts'] if error_logs_summary['sample_counts'] is not None else pd.DataFrame(columns=['message','count'])

# Keep outputs compact: show summary table and top lists (each limited already)
summary_table, metric_kpis, trace_kpis, log_kpis, error_messages

```
Out[1]:
```
```python
# Build a concise plain-English summary using previously loaded summary DataFrames
metric_top_list = ", ".join([f"{r.kpi_name}({int(r['count'])})" for _, r in metric_kpis.head(8).iterrows()])
trace_top_list = ", ".join([f"{r.trace_name}({int(r['count'])})" for _, r in trace_kpis.head(8).iterrows()])
log_top_list = ", ".join([f"{r.log_name}({int(r['count'])})" for _, r in log_kpis.iterrows()])

summary_text = (
    "Summary of telemetry files (UTC):\n"
    f"- metric.csv: {int(summary_table.loc[0,'total_rows'])} rows, {int(summary_table.loc[0,'unique_cmdb_count'])} unique cmdb_ids "
    f"(examples: {', '.join(summary_table.loc[0,'unique_cmdb_ids_truncated'][:6])}...), "
    f"time range {summary_table.loc[0,'earliest_ts_utc']} to {summary_table.loc[0,'latest_ts_utc']}. "
    f"Top metric KPIs: {metric_top_list}.\n"
    f"- trace.csv: {int(summary_table.loc[1,'total_rows'])} rows, {int(summary_table.loc[1,'unique_cmdb_count'])} unique cmdb_ids "
    f"(examples: {', '.join(summary_table.loc[1,'unique_cmdb_ids_truncated'][:6])}...), "
    f"time range {summary_table.loc[1,'earliest_ts_utc']} to {summary_table.loc[1,'latest_ts_utc']}. "
    f"Top trace features: {trace_top_list}.\n"
    f"- log.csv: {int(summary_table.loc[2,'total_rows'])} rows, {int(summary_table.loc[2,'unique_cmdb_count'])} unique cmdb_ids "
    f"(examples: {', '.join(summary_table.loc[2,'unique_cmdb_ids_truncated'][:6])}...), "
    f"time range {summary_table.loc[2,'earliest_ts_utc']} to {summary_table.loc[2,'latest_ts_utc']}. "
    f"Top log keys: {log_top_list}.\n"
    f"- error_logs.csv: {int(summary_table.loc[3,'total_rows'])} rows (empty), no timestamps.\n\n"
    "Notes: All telemetry covers the same short window (2024-01-20 05:18 to 05:42 UTC). "
    "Metric KPIs show frequent entries for cpu, workload, socket, and mem; trace data shows prominent root and service-specific duration/error metrics; logs include error counts. error_logs contains no entries."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file  total_rows  unique_cmdb_count                          unique_cmdb_ids_truncated           earliest_ts_utc             latest_ts_utc
0      metric.csv        1850                 12  [adservice, cartservice, checkoutservice, curr... 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
1       trace.csv        3800                  8  [checkoutservice, currencyservice, emailservic... 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
2         log.csv         460                 10  [adservice, cartservice, checkoutservice, curr... 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
3  error_logs.csv           0                  0                                                 []                       NaT                       NaT,      kpi_name  count
0         cpu    275
1    workload    275
2      socket    275
3         mem    275
4  latency-50    250
5  latency-90    250
6      diskio    200
7       error     50,                                       trace_name  count
0                  trace.from_root.duration_mean    175
1                   trace.from_root.duration_p95    175
2                     trace.from_root.error_rate    175
3                      trace.from_root.row_count    175
4           trace.from_checkoutservice.row_count    125
5          trace.from_checkoutservice.error_rate    125
6        trace.from_checkoutservice.duration_p95    125
7       trace.from_checkoutservice.duration_mean    125
8           trace.from_frontendservice.row_count    125
9          trace.from_frontendservice.error_rate    125
10       trace.from_frontendservice.duration_p95    125
11      trace.from_frontendservice.duration_mean    125
12      trace.to_productcatalogservice.row_count    100
13     trace.to_productcatalogservice.error_rate    100
14   trace.to_productcatalogservice.duration_p95    100
15  trace.to_productcatalogservice.duration_mean    100
16        trace.to_checkoutservice.duration_mean     75
17         trace.to_checkoutservice.duration_p95     75
18           trace.to_checkoutservice.error_rate     75
19            trace.to_checkoutservice.row_count     75,           log_name  count
0  log.error_count    230
1    log.row_count    230, Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Re-load metric.csv to ensure fresh state (re-using variable names is allowed)
metric_df = pd.read_csv("metric.csv")
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Filter to candidate services
cand_metric = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Calculate global P95 and total points (ALL-TIME) per (cmdb_id, kpi_name)
thresholds = (
    cand_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(total_points_alltime='count', global_p95=lambda s: s.quantile(0.95))
    .reset_index()
)

# Define incident window (UTC)
start = pd.Timestamp("2024-01-20 05:15:25", tz="UTC")
end   = pd.Timestamp("2024-01-20 05:45:25", tz="UTC")

# Incident-window rows for candidate services
incident_rows = cand_metric[(cand_metric['ts'] >= start) & (cand_metric['ts'] <= end)].copy()

# Incident stats per pair
incident_stats = (
    incident_rows
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(incident_points='count',
         max_value_in_incident='max',
         mean_value_in_incident='mean')
    .reset_index()
)

# Compute breaches within the incident window using global_p95 (thresholds computed from all-time)
incident_with_thresh = incident_rows.merge(
    thresholds[['cmdb_id', 'kpi_name', 'global_p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)
if not incident_with_thresh.empty:
    incident_with_thresh['is_breach'] = incident_with_thresh['value'] >= incident_with_thresh['global_p95']
    breach_agg = (
        incident_with_thresh[incident_with_thresh['is_breach']]
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(
            breach_count=('is_breach', 'sum'),
            earliest_breach_timestamp_utc=('ts', 'min'),
            latest_breach_timestamp_utc=('ts', 'max')
        )
        .reset_index()
    )
else:
    # empty frame with expected columns
    breach_agg = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'breach_count',
                                       'earliest_breach_timestamp_utc', 'latest_breach_timestamp_utc'])

# Merge all stats together (keep all pairs seen all-time in thresholds)
result = thresholds.merge(incident_stats, on=['cmdb_id', 'kpi_name'], how='left')
result = result.merge(breach_agg, on=['cmdb_id', 'kpi_name'], how='left')

# Fill missing numeric incident values with zeros or NaN where appropriate
result['incident_points'] = result['incident_points'].fillna(0).astype(int)
result['breach_count'] = result['breach_count'].fillna(0).astype(int)
result['max_value_in_incident'] = result['max_value_in_incident'].where(result['incident_points']>0, pd.NA)
result['mean_value_in_incident'] = result['mean_value_in_incident'].where(result['incident_points']>0, pd.NA)
result['earliest_breach_timestamp_utc'] = result['earliest_breach_timestamp_utc'].where(result['breach_count']>0, pd.NaT)
result['latest_breach_timestamp_utc'] = result['latest_breach_timestamp_utc'].where(result['breach_count']>0, pd.NaT)

# Select and order columns as requested
result_table = result[[
    'cmdb_id',
    'kpi_name',
    'total_points_alltime',
    'global_p95',
    'incident_points',
    'breach_count',
    'earliest_breach_timestamp_utc',
    'latest_breach_timestamp_utc',
    'max_value_in_incident',
    'mean_value_in_incident'
]]

# Sort by breach_count desc, then incident_points desc, then total_points_alltime desc
result_table = result_table.sort_values(
    by=['breach_count', 'incident_points', 'total_points_alltime'],
    ascending=[False, False, False]
).reset_index(drop=True)

# Keep top 20 rows (or fewer if less)
top20_result = result_table.head(20)

# Display the compact table
top20_result

```
Out[2]:
```
```python
summary_text = (
    "Summary of metric analysis for candidate services (incident window 2024-01-20 05:15:25–05:45:25 UTC):\n\n"
    "Top findings:\n"
    "1) Several (cmdb_id, kpi_name) pairs had all incident points >= their global P95 (incident_points=25, breach_count=25):\n"
    "   - checkoutservice / socket: 25/25 breached, max=9.0, mean=9.0, earliest breach 2024-01-20 05:18 UTC, latest 2024-01-20 05:42 UTC\n"
    "   - currencyservice / diskio: 25/25 breached (values 0), earliest 05:18, latest 05:42 UTC\n"
    "   - currencyservice / socket: 25/25 breached, values max=3.0, mean=3.0\n"
    "   - emailservice / diskio: 25/25 breached (values 0)\n"
    "   - productcatalogservice / diskio: 25/25 breached (values 0)\n"
    "   - productcatalogservice / socket: 25/25 breached (values 3.0)\n"
    "   - recommendationservice / diskio: 25/25 breached (values 0)\n\n"
    "2) Multiple other pairs show a small number of breaches (breach_count=2) in the incident window, e.g.,\n"
    "   - checkoutservice: cpu, latency-50, latency-90, mem, workload (each 2 breaches; breach times ~05:23–05:42 UTC)\n"
    "   - currencyservice: cpu, latency-50, latency-90, mem, workload (each 2 breaches)\n"
    "   - emailservice: cpu, latency-50, latency-90 (each 2 breaches)\n"
    "   For these the max and mean values are modestly higher than baseline (see table for exact numbers).\n\n"
    "Interpretation / recommendation:\n"
    "- The most widespread and consistent anomalies are in socket and diskio metrics across multiple services. In particular, checkoutservice.socket stands out with all incident samples at 9.0 (>= global P95).\n"
    "- Many diskio entries breach at 0 because the global P95 is 0 — consider whether that threshold is meaningful or should be relaxed before concluding disk I/O is problematic.\n"
    "- Action: prioritize investigating socket-related issues on checkoutservice and review network/socket error logs across services. If diskio thresholds remain 0, validate metric semantics before acting on disk I/O.\n"
)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_points_alltime    global_p95  incident_points  breach_count earliest_breach_timestamp_utc latest_breach_timestamp_utc  max_value_in_incident  mean_value_in_incident
0         checkoutservice      socket                    25  9.000000e+00               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00           9.000000e+00            9.000000e+00
1         currencyservice      diskio                    25  0.000000e+00               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00           0.000000e+00            0.000000e+00
2         currencyservice      socket                    25  3.000000e+00               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00           3.000000e+00            3.000000e+00
3            emailservice      diskio                    25  0.000000e+00               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00           0.000000e+00            0.000000e+00
4   productcatalogservice      diskio                    25  0.000000e+00               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00           0.000000e+00            0.000000e+00
5   productcatalogservice      socket                    25  3.000000e+00               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00           3.000000e+00            3.000000e+00
6   recommendationservice      diskio                    25  0.000000e+00               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00           0.000000e+00            0.000000e+00
7         checkoutservice         cpu                    25  5.278293e-01               25             2     2024-01-20 05:37:00+00:00   2024-01-20 05:42:00+00:00           6.445942e-01            4.437838e-01
8         checkoutservice  latency-50                    25  1.655277e-01               25             2     2024-01-20 05:31:00+00:00   2024-01-20 05:32:00+00:00           1.770237e-01            1.167131e-01
9         checkoutservice  latency-90                    25  2.510927e-01               25             2     2024-01-20 05:32:00+00:00   2024-01-20 05:33:00+00:00           2.966451e-01            2.286577e-01
10        checkoutservice         mem                    25  1.133912e+07               25             2     2024-01-20 05:24:00+00:00   2024-01-20 05:32:00+00:00           1.166077e+07            1.116374e+07
11        checkoutservice    workload                    25  1.159730e+00               25             2     2024-01-20 05:23:00+00:00   2024-01-20 05:41:00+00:00           1.192317e+00            9.768013e-01
12        currencyservice         cpu                    25  3.329828e+01               25             2     2024-01-20 05:34:00+00:00   2024-01-20 05:35:00+00:00           3.426300e+01            3.131812e+01
13        currencyservice  latency-50                    25  3.955138e-03               25             2     2024-01-20 05:35:00+00:00   2024-01-20 05:42:00+00:00           3.972964e-03            3.817972e-03
14        currencyservice  latency-90                    25  5.265121e-02               25             2     2024-01-20 05:35:00+00:00   2024-01-20 05:42:00+00:00           5.488046e-02            4.416239e-02
15        currencyservice         mem                    25  4.502382e+07               25             2     2024-01-20 05:40:00+00:00   2024-01-20 05:41:00+00:00           4.510938e+07            4.435353e+07
16        currencyservice    workload                    25  8.353978e+01               25             2     2024-01-20 05:34:00+00:00   2024-01-20 05:41:00+00:00           8.805785e+01            7.988534e+01
17           emailservice         cpu                    25  1.816472e+01               25             2     2024-01-20 05:31:00+00:00   2024-01-20 05:32:00+00:00           1.959131e+01            4.213903e+00
18           emailservice  latency-50                    25  4.906997e-02               25             2     2024-01-20 05:31:00+00:00   2024-01-20 05:32:00+00:00           6.402278e-02            8.234846e-03
19           emailservice  latency-90                    25  9.191735e-02               25             2     2024-01-20 05:31:00+00:00   2024-01-20 05:32:00+00:00           9.380280e-02            2.420076e-02```
```

In [ ]:
# In[3]:

import pandas as pd

# Re-load trace.csv (re-using variable names for efficiency)
trace_df = pd.read_csv("trace.csv")
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Restrict to candidate services
cand_trace = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()

# 1) Compute global P95 and total points all-time per (cmdb_id, trace_name)
thresholds = (
    cand_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(total_points_alltime='count',
         global_p95=lambda s: s.quantile(0.95))
    .reset_index()
)

# Incident window (UTC)
start = pd.Timestamp("2024-01-20 05:15:25", tz="UTC")
end   = pd.Timestamp("2024-01-20 05:45:25", tz="UTC")

# 2) Incident-window rows for candidate services
incident_rows = cand_trace[(cand_trace['ts'] >= start) & (cand_trace['ts'] <= end)].copy()

# Incident stats per pair
incident_stats = (
    incident_rows
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(incident_points='count',
         max_value_in_incident='max',
         mean_value_in_incident='mean')
    .reset_index()
)

# Compute breaches within the incident window using pre-computed global_p95
incident_with_thresh = incident_rows.merge(
    thresholds[['cmdb_id', 'trace_name', 'global_p95']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)
if not incident_with_thresh.empty:
    incident_with_thresh['is_breach'] = incident_with_thresh['value'] >= incident_with_thresh['global_p95']
    breach_agg = (
        incident_with_thresh[incident_with_thresh['is_breach']]
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(
            breach_count=('is_breach', 'sum'),
            earliest_breach_timestamp_utc=('ts', 'min'),
            latest_breach_timestamp_utc=('ts', 'max')
        )
        .reset_index()
    )
else:
    breach_agg = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'breach_count',
                                       'earliest_breach_timestamp_utc', 'latest_breach_timestamp_utc'])

# Merge all stats together (keep all pairs seen all-time)
result = thresholds.merge(incident_stats, on=['cmdb_id', 'trace_name'], how='left')
result = result.merge(breach_agg, on=['cmdb_id', 'trace_name'], how='left')

# Fill/normalize columns
result['incident_points'] = result['incident_points'].fillna(0).astype(int)
result['breach_count'] = result['breach_count'].fillna(0).astype(int)
result['max_value_in_incident'] = result['max_value_in_incident'].where(result['incident_points']>0, pd.NA)
result['mean_value_in_incident'] = result['mean_value_in_incident'].where(result['incident_points']>0, pd.NA)
result['earliest_breach_timestamp_utc'] = result['earliest_breach_timestamp_utc'].where(result['breach_count']>0, pd.NaT)
result['latest_breach_timestamp_utc'] = result['latest_breach_timestamp_utc'].where(result['breach_count']>0, pd.NaT)

# Select columns in requested order
result_table = result[[
    'cmdb_id',
    'trace_name',
    'total_points_alltime',
    'global_p95',
    'incident_points',
    'breach_count',
    'earliest_breach_timestamp_utc',
    'latest_breach_timestamp_utc',
    'max_value_in_incident',
    'mean_value_in_incident'
]]

# Sort by breach_count desc, then incident_points desc, then total_points_alltime desc
result_table = result_table.sort_values(
    by=['breach_count', 'incident_points', 'total_points_alltime'],
    ascending=[False, False, False]
).reset_index(drop=True)

# Keep top 20 rows
top20_trace_breaches = result_table.head(20)

# Display compact result
top20_trace_breaches

```
Out[3]:
```
```python
summary_text = (
    "Summary of trace analysis for candidate services (incident window 2024-01-20 05:15:25–05:45:25 UTC):\n\n"
    "- For every (cmdb_id, trace_name) pair shown, the global P95 computed over all available data is 0.0.\n"
    "- Because the incident values are also 0.0, every incident point meets the condition value >= global_P95, producing breach_count = 25 (incident_points = 25) for all pairs.\n"
    "- Practically, this means the breach results are driven by a zero-valued baseline (global P95 = 0) and do not indicate an increase in error rate or latency during the incident window.\n\n"
    "Recommendation:\n"
    "- Do not interpret these trace 'breaches' as real anomalies without adjusting the threshold logic. Consider using a stricter rule (e.g., value > global_P95, a higher percentile, or a baseline+delta), or inspect non-zero trace metrics such as duration/latency or row_count.\n"
    "- Correlate with metric findings (socket/diskio anomalies from metrics) and logs to find actionable signals rather than relying on error_rate traces where baseline is zero."
)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  total_points_alltime  global_p95  incident_points  breach_count earliest_breach_timestamp_utc latest_breach_timestamp_utc  max_value_in_incident  mean_value_in_incident
0         checkoutservice        trace.from_checkoutservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
1         checkoutservice        trace.from_frontendservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
2         checkoutservice                   trace.from_root.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
3         checkoutservice          trace.to_checkoutservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
4         checkoutservice          trace.to_currencyservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
5         checkoutservice             trace.to_emailservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
6         checkoutservice           trace.to_paymentservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
7         checkoutservice    trace.to_productcatalogservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
8         currencyservice        trace.from_checkoutservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
9         currencyservice        trace.from_frontendservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
10        currencyservice                   trace.from_root.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
11           emailservice        trace.from_checkoutservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
12           emailservice                   trace.from_root.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
13  productcatalogservice        trace.from_checkoutservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
14  productcatalogservice        trace.from_frontendservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
15  productcatalogservice  trace.from_recommendationservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
16  productcatalogservice                   trace.from_root.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
17  recommendationservice        trace.from_frontendservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
18  recommendationservice  trace.from_recommendationservice.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0
19  recommendationservice                   trace.from_root.error_rate                    25         0.0               25            25     2024-01-20 05:18:00+00:00   2024-01-20 05:42:00+00:00                    0.0                     0.0```
```

In [ ]:
# In[4]:

import pandas as pd

# Re-load log and error_logs to ensure fresh state (kernel is stateful but reload is safe)
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes
for df in (log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    else:
        df['ts'] = pd.NaT

# Candidate services and incident window (UTC)
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]
start = pd.Timestamp("2024-01-20 05:15:25", tz="UTC")
end   = pd.Timestamp("2024-01-20 05:45:25", tz="UTC")

# Filter log rows to candidates and incident window
logs_cand = log_df[log_df['cmdb_id'].isin(candidates)].copy()
logs_incident = logs_cand[(logs_cand['ts'] >= start) & (logs_cand['ts'] <= end)].copy()

# Ensure 'value' exists and is numeric
if 'value' in logs_incident.columns:
    logs_incident['value'] = pd.to_numeric(logs_incident['value'], errors='coerce')
else:
    logs_incident['value'] = pd.NA

# Group by (cmdb_id, log_name) and compute requested aggregates
grouped = (
    logs_incident
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(
        incident_points=('value', 'count'),
        sum_value=('value', 'sum'),
        max_value=('value', 'max'),
        earliest_timestamp_utc=('ts', 'min'),
        latest_timestamp_utc=('ts', 'max')
    )
    .reset_index()
)

# Replace possible NA sums with 0 for sorting clarity
grouped['sum_value'] = grouped['sum_value'].fillna(0)

# Sort by sum_value descending and limit to top 20
grouped_top20 = grouped.sort_values(by='sum_value', ascending=False).head(20).reset_index(drop=True)

# Provide a quick sample (top 10) of raw log rows for candidate services within the incident window
# Sort sample by timestamp ascending for inspection
sample_raw_top10 = logs_incident.sort_values(by='ts').head(10)[['ts', 'timestamp', 'cmdb_id', 'log_name', 'value']].reset_index(drop=True)

# Handle error_logs: total row count and top 20 rows (timestamp, cmdb_id, message) or empty note
error_count = len(error_logs_df)
if error_count > 0:
    # ensure ts parsed and include requested columns, limit 20
    error_sample = error_logs_df.sort_values(by='ts').head(20)[['ts', 'timestamp', 'cmdb_id', 'message']].reset_index(drop=True)
    error_result = error_sample
else:
    error_result = "error_logs.csv is empty (0 rows)."

# Display compact results: grouped summary, sample raw logs, error count, and error_result (DataFrame or message)
grouped_top20, sample_raw_top10, error_count, error_result

```
Out[4]:
```
```python
summary = (
    "Log and error-log summary for candidate services during 2024-01-20 05:15:25–05:45:25 UTC:\n\n"
    "1) Aggregated log groups (top by sum_value):\n"
    "   - currencyservice / log.row_count: incident_points=25, sum=57575, max=2632 (earliest 05:18, latest 05:42 UTC)\n"
    "   - recommendationservice / log.row_count: incident_points=25, sum=13539, max=602 (05:18–05:42 UTC)\n"
    "   - checkoutservice / log.row_count: incident_points=25, sum=2127, max=108 (05:18–05:42 UTC)\n"
    "   - emailservice / log.row_count: incident_points=25, sum=709, max=36 (05:18–05:42 UTC)\n"
    "   - For all candidate services, log.error_count has 25 points each but sum=0 and max=0 (no recorded error counts).\n\n"
    "2) Raw sample (first 10 rows) confirms per-minute log entries starting at 2024-01-20 05:18:00 UTC with values like row_count and error_count (error_count=0).\n\n"
    "3) error_logs.csv: empty (0 rows).\n\n"
    "Interpretation / next steps:\n"
    "- No logged errors were recorded in the incident window (log.error_count = 0 and error_logs empty).\n"
    "- currencyservice and recommendationservice show much higher log row counts (traffic) than other services—these are candidates to correlate with metric/trace anomalies.\n"
    "- Recommend correlating these high row_count services with the metric-level socket/diskio anomalies and trace anomalies already identified to find root cause."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id         log_name  incident_points  sum_value  max_value    earliest_timestamp_utc      latest_timestamp_utc
0        currencyservice    log.row_count               25      57575       2632 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
1  recommendationservice    log.row_count               25      13539        602 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
2        checkoutservice    log.row_count               25       2127        108 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
3           emailservice    log.row_count               25        709         36 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
4        currencyservice  log.error_count               25          0          0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
5        checkoutservice  log.error_count               25          0          0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
6           emailservice  log.error_count               25          0          0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
7  recommendationservice  log.error_count               25          0          0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00,                          ts   timestamp                cmdb_id         log_name  value
0 2024-01-20 05:18:00+00:00  1705727880        checkoutservice  log.error_count      0
1 2024-01-20 05:18:00+00:00  1705727880        checkoutservice    log.row_count     36
2 2024-01-20 05:18:00+00:00  1705727880        currencyservice  log.error_count      0
3 2024-01-20 05:18:00+00:00  1705727880        currencyservice    log.row_count   1274
4 2024-01-20 05:18:00+00:00  1705727880           emailservice  log.error_count      0
5 2024-01-20 05:18:00+00:00  1705727880           emailservice    log.row_count     12
6 2024-01-20 05:18:00+00:00  1705727880  recommendationservice  log.error_count      0
7 2024-01-20 05:18:00+00:00  1705727880  recommendationservice    log.row_count    324
8 2024-01-20 05:19:00+00:00  1705727940  recommendationservice    log.row_count    558
9 2024-01-20 05:19:00+00:00  1705727940  recommendationservice  log.error_count      0, 0, 'error_logs.csv is empty (0 rows).')```
```

In [ ]:
# In[5]:

import pandas as pd

# Re-load trace.csv (reusing variable names is fine)
trace_df = pd.read_csv("trace.csv")
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services and incident window (UTC)
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]
start = pd.Timestamp("2024-01-20 05:15:25", tz="UTC")
end   = pd.Timestamp("2024-01-20 05:45:25", tz="UTC")

# Filter to candidates and incident window
cand_trace_incident = trace_df[
    trace_df['cmdb_id'].isin(candidates) &
    (trace_df['ts'] >= start) & (trace_df['ts'] <= end)
].copy()

# Ensure numeric values
cand_trace_incident['value'] = pd.to_numeric(cand_trace_incident['value'], errors='coerce')

# Group and aggregate per (cmdb_id, trace_name)
agg = (
    cand_trace_incident
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        incident_points=('value', 'count'),
        sum_value=('value', 'sum'),
        mean_value=('value', 'mean'),
        max_value=('value', 'max'),
        earliest_timestamp_utc=('ts', 'min'),
        latest_timestamp_utc=('ts', 'max')
    )
    .reset_index()
)

# Compact output 1: top 20 by sum_value descending
top20_traces = agg.sort_values(by='sum_value', ascending=False).head(20).reset_index(drop=True)

# Compact output 2: focused traces matching criteria
pattern = r"(row_count|duration_mean|to_checkoutservice|from_checkoutservice)"
focused = agg[agg['trace_name'].str.contains(pattern, regex=True, na=False)].copy()
focused_traces = focused.sort_values(by='sum_value', ascending=False).head(20).reset_index(drop=True)

# Display compact results
top20_traces, focused_traces

```
Out[5]:
```
Summary of trace activity for candidate services during 2024-01-20 05:15:25–05:45:25 UTC

Top traces by total traffic (sum_value) — top 5 highlights
- productcatalogservice / trace.from_frontendservice.row_count: incident_points=25, sum=88,177, mean≈3,527, max=3,935 (05:18–05:42 UTC)
- currencyservice / trace.from_frontendservice.row_count: incident_points=25, sum=55,138, mean≈2,205, max=2,534 (05:18–05:42 UTC)
- recommendationservice / trace.from_frontendservice.row_count: incident_points=25, sum=13,539, mean≈542, max=602 (05:18–05:42 UTC)
- productcatalogservice / trace.from_recommendationservice.row_count: incident_points=25, sum=13,539, mean≈542, max=602 (05:18–05:42 UTC)
- recommendationservice / trace.to_productcatalogservice.row_count: incident_points=25, sum=13,539, mean≈542, max=602 (05:18–05:42 UTC)

Focused traces (row_count and checkout-related) — key observations
- High row_count traffic flows involve productcatalogservice, currencyservice, recommendationservice, and checkoutservice. Examples:
  - checkoutservice trace.from_checkoutservice.row_count and trace.to_checkoutservice.row_count: incident_points=25, sum≈5,979, mean≈239, max=308.
  - currencyservice trace.from_checkoutservice.row_count and checkoutservice trace.to_currencyservice.row_count: incident_points=25, sum≈2,434, mean≈97, max=131.
  - Many to/from checkoutservice and to/from productcatalogservice row_count metrics show nontrivial traffic across the window (all earliest 05:18, latest 05:42 UTC).
- All aggregated groups use 25 incident points (data sampled every minute across the ~25-minute window).

Interpretation / next steps
- The heaviest trace traffic is toward productcatalogservice and from frontend → productcatalogservice, with currencyservice and recommendationservice also showing substantial row_count volume. Checkoutservice also has notable inter-service row_count activity.
- These trace volumes align with earlier log/metric findings that showed high row_count and traffic for currencyservice, recommendationservice and productcatalogservice. Correlate these heavy trace flows with the metric anomalies (socket/diskio) and logs to narrow the root cause—e.g., investigate socket/network handling on checkoutservice and productcatalogservice and whether high traffic patterns coincide with socket/diskio signals.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                  trace_name  incident_points  sum_value  mean_value  max_value    earliest_timestamp_utc      latest_timestamp_utc
0   productcatalogservice        trace.from_frontendservice.row_count               25    88177.0     3527.08     3935.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
1         currencyservice        trace.from_frontendservice.row_count               25    55138.0     2205.52     2534.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
2   recommendationservice        trace.from_frontendservice.row_count               25    13539.0      541.56      602.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
3   productcatalogservice  trace.from_recommendationservice.row_count               25    13539.0      541.56      602.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
4   recommendationservice    trace.to_productcatalogservice.row_count               25    13539.0      541.56      602.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
5   recommendationservice    trace.to_recommendationservice.row_count               25    13539.0      541.56      602.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
6   recommendationservice  trace.from_recommendationservice.row_count               25    13539.0      541.56      602.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
7         checkoutservice        trace.from_checkoutservice.row_count               25     5979.0      239.16      308.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
8         checkoutservice          trace.to_checkoutservice.row_count               25     5979.0      239.16      308.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
9         checkoutservice                   trace.from_root.row_count               25     3010.0      120.40      155.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
10        currencyservice        trace.from_checkoutservice.row_count               25     2434.0       97.36      131.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
11        checkoutservice          trace.to_currencyservice.row_count               25     2434.0       97.36      131.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
12  productcatalogservice        trace.from_checkoutservice.row_count               25     1725.0       69.00       96.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
13        checkoutservice    trace.to_productcatalogservice.row_count               25     1725.0       69.00       96.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
14        checkoutservice           trace.to_paymentservice.row_count               25      709.0       28.36       36.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
15           emailservice        trace.from_checkoutservice.row_count               25      709.0       28.36       36.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
16        checkoutservice             trace.to_emailservice.row_count               25      709.0       28.36       36.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
17        checkoutservice        trace.from_frontendservice.row_count               25      709.0       28.36       36.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
18  recommendationservice                   trace.from_root.row_count               25      576.0       23.04       24.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
19  productcatalogservice                   trace.from_root.row_count               25      576.0       23.04       24.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00,                   cmdb_id                                  trace_name  incident_points  sum_value  mean_value  max_value    earliest_timestamp_utc      latest_timestamp_utc
0   productcatalogservice        trace.from_frontendservice.row_count               25    88177.0     3527.08     3935.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
1         currencyservice        trace.from_frontendservice.row_count               25    55138.0     2205.52     2534.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
2   recommendationservice        trace.from_frontendservice.row_count               25    13539.0      541.56      602.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
3   recommendationservice    trace.to_recommendationservice.row_count               25    13539.0      541.56      602.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
4   recommendationservice    trace.to_productcatalogservice.row_count               25    13539.0      541.56      602.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
5   recommendationservice  trace.from_recommendationservice.row_count               25    13539.0      541.56      602.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
6   productcatalogservice  trace.from_recommendationservice.row_count               25    13539.0      541.56      602.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
7         checkoutservice        trace.from_checkoutservice.row_count               25     5979.0      239.16      308.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
8         checkoutservice          trace.to_checkoutservice.row_count               25     5979.0      239.16      308.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
9         checkoutservice                   trace.from_root.row_count               25     3010.0      120.40      155.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
10        currencyservice        trace.from_checkoutservice.row_count               25     2434.0       97.36      131.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
11        checkoutservice          trace.to_currencyservice.row_count               25     2434.0       97.36      131.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
12        checkoutservice    trace.to_productcatalogservice.row_count               25     1725.0       69.00       96.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
13  productcatalogservice        trace.from_checkoutservice.row_count               25     1725.0       69.00       96.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
14        checkoutservice             trace.to_emailservice.row_count               25      709.0       28.36       36.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
15        checkoutservice           trace.to_paymentservice.row_count               25      709.0       28.36       36.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
16           emailservice        trace.from_checkoutservice.row_count               25      709.0       28.36       36.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
17        checkoutservice        trace.from_frontendservice.row_count               25      709.0       28.36       36.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
18  recommendationservice                   trace.from_root.row_count               25      576.0       23.04       24.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00
19  productcatalogservice                   trace.from_root.row_count               25      576.0       23.04       24.0 2024-01-20 05:18:00+00:00 2024-01-20 05:42:00+00:00)```
```